# SeoulMate RAG / MCP - 20개 질문 유형 테스트

이 노트북은 **파일 하나로 실행**된다. 실제 프로젝트의
`rag.py` / `query_policy.py` / `weather_reranker.py` 로직을 그대로 태우고,
DB 접근과 임베딩만 노트북 안의 **가상 식당 22곳**으로 목킹한다.

- 카테고리 추출 / 벡터 RRF / 카테고리 부스팅 / 하드필터 / 정렬 / 날씨 재랭킹 = **실제 코드**
- 파서(GPT)는 이 환경에서 흉내낼 수 없으므로, GPT 출력에 해당하는
  `search_query` / `filters` / `themes` 를 각 케이스에 직접 넣는다.

> 이 노트북을 backend 폴더(또는 그 하위) 안에 두고 실행하라. backend 를 자동 탐색한다.
> DB/OpenAI 키는 **필요 없다** (전부 목킹).

| 절 | 내용 |
|---|---|
| 1 | 하네스 (가상 DB + 목킹) |
| 2 | 20개 케이스 정의 |
| 3 | 전체 실행 + 결과 요약표 |
| 4 | 케이스별 점수 분해 상세 |
| 5 | 추가 진단 A - 카테고리 부스팅이 순위를 바꾸나 |
| 6 | 추가 진단 B - 날씨 재랭킹이 순위를 뒤집나 |
| 7 | 추가 진단 C - '서울 홍대' 지역 오인 |

## 1. 하네스 (가상 DB + 목킹)

In [1]:
# ============================================================
# 테스트 하네스 - 실제 로직을 태우고 DB/임베딩만 목킹
# ============================================================
# 실제 프로젝트 모듈을 import 한다. (backend 를 자동 탐색)
import sys, copy
from pathlib import Path
from contextlib import contextmanager

BACKEND_DIR = None
for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "services" / "rag.py").exists() and (base / "services" / "weather_reranker.py").exists():
        BACKEND_DIR = base
        break
if BACKEND_DIR is None:
    # 필요하면 직접 지정: BACKEND_DIR = Path(r"C:\Users\user\Desktop\seoulmate\SeoulMate\backend")
    raise RuntimeError("backend 를 찾지 못했습니다. 이 노트북을 backend 안에 두거나 BACKEND_DIR 를 지정하세요.")
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

import services.rag as R
from services.rag import search_restaurants_structured, build_restaurant_search_plan
from services.weather_reranker import rerank_with_weather
from services.location import haversine_km
from schemas.structured_query import StructuredTravelQuery, StructuredQueryTask

print("backend:", BACKEND_DIR)


# ── 가상 식당 DB (다양한 카테고리/지역/속성/가격) ──
def _R(rid, name, cat, kakao, lat, lng, rating, reviews, price, hours,
       parking=False, pets=False, kids=False, group=False, room=False,
       baby=False, disabled=False, desc="", text=""):
    return dict(id=rid, name=name, category=cat, category_kakao=kakao,
                rating=rating, review_count=reviews, hours=hours,
                description=desc, description_kakao="", address="서울", image=None,
                lat=lat, lng=lng, menu_price_min=max(price - 5000, 3000),
                menu_price_median=price, has_parking=parking, allows_pets=pets,
                has_kids_menu=kids, has_group_seating=group, has_private_room=room,
                has_baby_chair=baby, has_disabled_access=disabled, _text=text)

DB = [
    _R(1,"홍대반점","중식","중국 요리",37.5565,126.9240,4.3,210,15000,"매일 11:00-22:00",
       parking=True,room=True,desc="조용하고 아늑한 중식",text="중식 짜장면 짬뽕 탕수육 조용한 아늑한 룸"),
    _R(2,"성화루","중식","중국 요리",37.5558,126.9255,4.6,520,28000,"매일 11:30-21:30",
       group=True,desc="루프탑 고급 중식",text="중식 코스요리 마라 시원한 냉채 루프탑 테라스 단체석"),
    _R(3,"old홍대반점","중식","중국 요리",37.5540,126.9300,4.0,55,12000,"매일 10:00-21:00",
       parking=True,desc="가성비 노포",text="중식 백반 저렴한 가성비 주차"),
    _R(4,"홍대한상","한식","한식",37.5567,126.9235,4.5,340,22000,"매일 11:00-22:00",
       parking=True,room=True,kids=True,desc="조용한 한정식",text="한식 한정식 조용한 룸 주차 아이 어린이"),
    _R(5,"분식왕","분식","한식",37.5561,126.9248,4.2,890,8000,"매일 10:00-23:00",
       desc="떡볶이 맛집",text="분식 떡볶이 순대 튀김 김밥 매운"),
    _R(6,"무브먼트커피","카페","카페",37.5559,126.9252,4.7,1200,9000,"매일 09:00-23:00",
       pets=True,desc="브런치 카페",text="카페 커피 디저트 브런치 분위기 반려동물 애견"),
    _R(7,"성수야경카페","카페","카페",37.5450,127.0560,4.8,640,12000,"매일 10:00-24:00",
       desc="성수 루프탑 카페",text="카페 커피 루프탑 테라스 감성 야경 디저트"),
    _R(8,"스시강남","일식","일본 요리",37.4982,127.0270,4.6,410,45000,"매일 12:00-22:00",
       room=True,disabled=True,desc="오마카세",text="일식 스시 초밥 오마카세 룸 휠체어 장애인"),
    _R(9,"라멘야","일식","일본 요리",37.4975,127.0285,4.3,730,11000,"매일 11:00-21:00",
       desc="돈코츠 라멘",text="일식 라멘 우동 돈코츠 따뜻한 국물"),
    _R(10,"파스타공방","이탈리안","이탈리아 요리",37.4988,127.0262,4.4,300,26000,"매일 11:30-22:00",
       kids=True,baby=True,group=True,desc="가족 이탈리안",text="이탈리안 파스타 피자 리조또 아이 유아 단체"),
    _R(11,"더스테이크","스테이크","스테이크하우스",37.4979,127.0290,4.7,260,68000,"매일 17:00-23:00",
       parking=True,room=True,disabled=True,desc="프리미엄 스테이크",text="스테이크 소고기 와인 룸 주차 휠체어"),
    _R(12,"숯불갈비","바베큐","바베큐",37.4970,127.0300,4.5,980,32000,"매일 16:00-24:00",
       parking=True,group=True,desc="한우 갈비",text="바베큐 갈비 삼겹살 고기 소고기 주차 단체석 회식"),
    _R(13,"타코로코","멕시코 요리","멕시코 요리",37.5344,126.9948,4.2,180,18000,"매일 11:00-23:00",
       pets=True,desc="캐주얼 멕시칸",text="멕시칸 타코 부리또 나초 반려동물"),
    _R(14,"방콕키친","태국음식","타이 요리",37.5340,126.9955,4.4,220,19000,"매일 11:30-22:00",
       desc="정통 태국",text="태국 팟타이 똠양꿍 시원한 쌀국수 매운"),
    _R(15,"사이공","베트남 요리","베트남 요리",37.5348,126.9940,4.3,340,13000,"매일 10:30-21:30",
       kids=True,desc="쌀국수 전문",text="베트남 쌀국수 반미 시원한 국물 아이"),
    _R(16,"델리인디아","인도 요리","인도 요리",37.5352,126.9952,4.1,130,21000,"매일 11:00-22:00",
       desc="북인도 커리",text="인도 커리 난 탄두리 매운"),
    _R(17,"종로삼계탕","삼계탕","한식",37.5722,126.9790,4.4,560,17000,"매일 10:00-21:00",
       disabled=True,desc="보양 삼계탕",text="한식 삼계탕 백숙 따뜻한 보양 휠체어"),
    _R(18,"노량진회","해산물","해산물",37.5720,126.9800,4.2,410,40000,"매일 11:00-23:00",
       group=True,parking=True,desc="활어회 물회",text="해산물 회 물회 시원한 조개 주차 단체"),
    _R(19,"여의도중식","중식","중국 요리",37.5216,126.9241,4.9,800,25000,"매일 11:00-22:00",
       parking=True,desc="최고 평점 중식(멀다)",text="중식 짜장 짬뽕 고급 주차"),
    _R(20,"저평점중식","중식","중국 요리",37.5562,126.9245,3.4,40,14000,"매일 11:00-20:00",
       desc="평범한 중식",text="중식 짜장면 짬뽕"),
    _R(21,"강남브런치","카페","카페",37.4985,127.0275,4.5,500,15000,"매일 08:00-22:00",
       kids=True,baby=True,desc="가족 브런치",text="카페 브런치 커피 디저트 아이 유아 팬케이크"),
    _R(22,"24시국밥","한식","한식",37.5718,126.9785,4.0,300,9000,"매일 00:00-24:00",
       desc="24시 국밥",text="한식 국밥 따뜻한 해장 24시"),
]
META = {r["id"]: r for r in DB}


def _tok(s):
    return set(str(s or "").replace(",", " ").split())

def _sim(qtext, dtext):
    q, d = _tok(qtext), _tok(dtext)
    return len(q & d) / len(q | d) if q and d else 0.0

class _Vec:
    def __init__(self, text): self.text = text

def _fake_embedding(text): return _Vec(text)

def _fake_ids_within_radius(cursor, lat, lng, radius_km, table):
    return [rid for rid, m in META.items() if haversine_km(lat, lng, m["lat"], m["lng"]) <= radius_km]

def _rank(vec, ids):
    return sorted(ids, key=lambda rid: -_sim(vec.text, META[rid]["_text"]))

def _fake_vector_search(cursor, suffix, vec, rvec, mvec, ids):
    ids = ids if ids is not None else list(META.keys())
    rest = {rid: {"rank": i, "similarity": max(_sim(vec.text, META[rid]["_text"]), 0.01)}
            for i, rid in enumerate(_rank(vec, ids))}
    review = {}
    for i, rid in enumerate(_rank(rvec, ids)):
        s = _sim(rvec.text, META[rid]["_text"])
        if s > 0:
            review[rid] = [{"rank": i, "similarity": s, "content": META[rid]["_text"]}]
    menu = {}
    for i, rid in enumerate(_rank(mvec, ids)):
        s = _sim(mvec.text, META[rid]["_text"])
        if s > 0:
            menu[rid] = [{"rank": i, "similarity": s, "name": META[rid]["name"], "is_main": (i == 0)}]
    return rest, review, menu

class _Cur:
    def __init__(self): self._rows = []
    def execute(self, q, params=None):
        if "FROM restaurant_" in q and "id = ANY" in q:
            ids = params[0] if params else []
            self._rows = [META[i] for i in ids if i in META]
        else:
            self._rows = []
    def fetchall(self): return list(self._rows)
    def fetchone(self): return self._rows[0] if self._rows else None
    def __enter__(self): return self
    def __exit__(self, *a): return False

class _Conn:
    def cursor(self, **k): return _Cur()
    def __enter__(self): return self
    def __exit__(self, *a): return False

@contextmanager
def _fake_connection():
    yield _Conn()

def _fake_menu_features(suffix, ids):
    out = {}
    for rid in ids:
        t = META[rid]["_text"]
        out[rid] = {"has_warm_menu": any(w in t for w in ["따뜻한","국물","국밥","삼계탕","라멘"]),
                    "has_cool_menu": any(w in t for w in ["시원한","냉면","냉채","물회","쌀국수"]),
                    "warm_menu_matches": ["국물"], "cool_menu_matches": ["냉면"]}
    return out

# 실제 rag 모듈의 저수준 DB/임베딩 함수만 목으로 교체
R._connection = _fake_connection
R._embedding = _fake_embedding
R._ids_within_radius = _fake_ids_within_radius
R._vector_search = _fake_vector_search
R._load_weather_menu_features = _fake_menu_features

CUR_LAT, CUR_LNG = 37.5665, 126.9780   # 사용자 현재 위치(서울시청)

def build_query(question, sq=None, loc=None, themes=None, filters=None):
    task = StructuredQueryTask(task_id="t1", domain="restaurant",
                               search_query=sq or question, themes=themes or [])
    f = {"location": loc} if loc else {}
    if filters:
        f.update(filters)
    parsed = StructuredTravelQuery(intent="single_place_recommendation",
        original_question=question, normalized_question=question, tasks=[task], filters=f)
    return parsed, task

def run_search(question, sq=None, loc=None, themes=None, filters=None, top_n=10):
    parsed, task = build_query(question, sq, loc, themes, filters)
    return parsed, task, R.search_restaurants_structured(
        parsed, task, current_lat=CUR_LAT, current_lng=CUR_LNG, top_n=top_n)

def names(res):
    return [c["name"] for c in res["candidates"]]

def base_of(c):
    b = c["breakdown"]
    if "base_score" in b:
        return b["base_score"]
    return b.get("restaurant_rrf", 0) + b.get("review_rrf", 0) + b.get("menu_rrf", 0)

def score_table(res):
    import pandas as pd
    rows = []
    for i, c in enumerate(res["candidates"], 1):
        b = c["breakdown"]
        rows.append({"#": i, "이름": c["name"],
                     "식당RRF": round(b.get("restaurant_rrf", 0), 5),
                     "리뷰RRF": round(b.get("review_rrf", 0), 5),
                     "메뉴RRF": round(b.get("menu_rrf", 0), 5),
                     "base": round(base_of(c), 5),
                     "status": b.get("category_status", "-"),
                     "부스트": round(b.get("category_boost", b.get("category_adjustment", 0)), 5),
                     "최종": round(c["score"], 5),
                     "평점": c["rating"], "거리km": None if c["distance_km"] is None else round(c["distance_km"], 2),
                     "주차": c["has_parking"], "중앙가": c["menu_price_median"]})
    return pd.DataFrame(rows)

print("하네스 준비 완료 - 식당", len(DB), "곳 / run_search, score_table 사용 가능")

backend: C:\Users\user\Desktop\seoulmate\SeoulMate\backend
하네스 준비 완료 - 식당 22 곳 / run_search, score_table 사용 가능


## 2. 20개 케이스 정의

각 케이스: (id, 유형, 질문, search_query, 지역, themes, filters, checks, weather).
`checks` 는 (설명, `res -> bool`) 목록. `weather` 가 있으면 재랭킹까지 적용.

In [ ]:
CASES = []

def case(cid, ctype, question, sq=None, loc=None, themes=None, filters=None, checks=None, weather=None):
    CASES.append(dict(id=cid, type=ctype, question=question, sq=sq, loc=loc,
                      themes=themes, filters=filters, checks=checks or [], weather=weather))

# 1) 기본 카테고리+지역
case(1,"카테고리+지역","홍대에서 중식당 추천해줘","홍대 중식당","홍대",
     checks=[("중국 요리로 추출", lambda p,t,r: r["extracted_category"]=="중국 요리"),
             ("mismatch 없음", lambda p,t,r: all(c["breakdown"]["category_status"]!="mismatch" for c in r["candidates"])),
             ("반경 밖 여의도중식 제외", lambda p,t,r: "여의도중식" not in names(r))])
# 2) 분위기 theme
case(2,"분위기 theme","홍대 조용한 중식당","홍대 조용한 중식당","홍대",themes=["조용한"],
     checks=[("조용한 홍대반점 상위2", lambda p,t,r: "홍대반점" in names(r)[:2])])
# 3) 메뉴명(떡볶이)
case(3,"메뉴명","홍대 떡볶이 맛집","홍대 떡볶이 맛집","홍대",
     checks=[("분식왕 포함", lambda p,t,r: "분식왕" in names(r)),
             ("분식왕 1위", lambda p,t,r: names(r)[0]=="분식왕")])
# 4) 평점 하한
case(4,"평점 하한","홍대에서 평점 4.0 이상 중식당","홍대 중식당","홍대",filters={"min_rating":4.0},
     checks=[("저평점중식 제외", lambda p,t,r: "저평점중식" not in names(r)),
             ("모두 평점>=4.0", lambda p,t,r: all(c["rating"]>=4.0 for c in r["candidates"]))])
# 5) 주차
case(5,"시설:주차","홍대에서 주차 가능한 중식당","홍대 주차 가능한 중식당","홍대",
     checks=[("모두 주차 가능", lambda p,t,r: all(c["has_parking"] is True for c in r["candidates"])),
             ("주차 없는 성화루 제외", lambda p,t,r: "성화루" not in names(r))])
# 6) 룸
case(6,"시설:룸","강남에서 룸 있는 스테이크집","강남 룸 스테이크","강남",
     checks=[("스테이크하우스 추출", lambda p,t,r: r["extracted_category"]=="스테이크하우스"),
             ("모두 룸 보유", lambda p,t,r: all(c["has_private_room"] is True for c in r["candidates"]) if r["candidates"] else True)])
# 7) 예산
case(7,"예산 상한","강남에서 3만원 이하 일식당","강남 일식당","강남",filters={"budget_max_krw":30000},
     checks=[("모두 중앙가<=30000", lambda p,t,r: all(c["menu_price_median"]<=30000 for c in r["candidates"])),
             ("스시강남(45000) 제외", lambda p,t,r: "스시강남" not in names(r))])
# 8) 키즈
case(8,"시설:키즈","강남에서 아이랑 갈 이탈리안","강남 아이 이탈리안","강남",filters={"required_features":["키즈 메뉴"]},
     checks=[("파스타공방 포함", lambda p,t,r: "파스타공방" in names(r)),
             ("모두 키즈 보유", lambda p,t,r: all(c["has_kids_menu"] is True for c in r["candidates"]) if r["candidates"] else True)])
# 9) 접근성
case(9,"시설:접근성","종로에서 휠체어 되는 한식당","종로 휠체어 한식당","종로",filters={"accessibility":["휠체어"]},
     checks=[("모두 장애인접근", lambda p,t,r: all(c["has_disabled_access"] is True for c in r["candidates"]) if r["candidates"] else True)])
# 10) 영문
case(10,"영문","Find a chinese restaurant in Hongdae","Hongdae chinese restaurant","홍대",
     checks=[("중국 요리 추출(영문)", lambda p,t,r: r["extracted_category"]=="중국 요리"),
             ("후보 존재", lambda p,t,r: len(r["candidates"])>0)])
# 11) accessible 오탐
case(11,"오탐:accessible","Find a cafe easily accessible from Hongdae","Hongdae cafe","홍대",
     checks=[("카페 추출", lambda p,t,r: r["extracted_category"]=="카페"),
             ("장애인접근 오탐 아님(후보 존재)", lambda p,t,r: len(r["candidates"])>0)])
# 12) 태국
case(12,"카테고리:태국","이태원 태국음식점 추천","이태원 태국음식점","이태원",
     checks=[("타이 요리 추출", lambda p,t,r: r["extracted_category"]=="타이 요리"),
             ("방콕키친 포함", lambda p,t,r: "방콕키친" in names(r))])
# 13) 바베큐
case(13,"카테고리:바베큐","강남 단체 회식 고깃집","강남 단체 고깃집","강남",filters={"required_features":["단체석"]},
     checks=[("바베큐 추출", lambda p,t,r: r["extracted_category"]=="바베큐"),
             ("숯불갈비 포함", lambda p,t,r: "숯불갈비" in names(r)),
             ("단체석 보유만", lambda p,t,r: all(c["has_group_seating"] is True for c in r["candidates"]) if r["candidates"] else True)])
# 14) 카테고리 없음
case(14,"카테고리 없음","홍대 맛집 추천","홍대 맛집","홍대",
     checks=[("카테고리 미지정", lambda p,t,r: r["extracted_category"] is None),
             ("여러 카테고리 혼재", lambda p,t,r: len({c["category_kakao"] for c in r["candidates"]})>=2)])
# 15) 반경
case(15,"반경 필터","홍대 중식당","홍대 중식당","홍대",
     checks=[("평점9.9 여의도중식 반경밖 제외", lambda p,t,r: "여의도중식" not in names(r))])
# 16) 평점 선호(모호)
case(16,"평점 선호(모호)","홍대에서 평점 좋은 중식당","홍대 중식당","홍대",themes=["평점 좋은"],
     checks=[("후보 존재", lambda p,t,r: len(r["candidates"])>0)])
# 17) 카페
case(17,"카테고리:카페","강남 브런치 카페","강남 브런치 카페","강남",
     checks=[("카페 추출", lambda p,t,r: r["extracted_category"]=="카페"),
             ("강남브런치 포함", lambda p,t,r: "강남브런치" in names(r))])
# 18) 해산물
case(18,"카테고리:해산물","종로 회 물회 잘하는 집","종로 회 물회","종로",
     checks=[("해산물 추출", lambda p,t,r: r["extracted_category"]=="해산물"),
             ("노량진회 포함", lambda p,t,r: "노량진회" in names(r))])
# 19) 날씨:비
case(19,"날씨 재랭킹:비","내일 비 오는데 홍대 중식당","홍대 중식당","홍대",
     weather={"available":True,"condition":"rain","temperature_c":18,"wind_speed_mps":9.0},
     checks=[("모두 rag_score 기록", lambda p,t,r: all(c.get("rag_score") is not None for c in r["candidates"])),
             ("weather_score in [0,1]", lambda p,t,r: all(0<=(c.get("weather_score") or 0)<=1 for c in r["candidates"]))])
# 20) 날씨:폭염
case(20,"날씨 재랭킹:폭염","오늘 더운데 이태원 쌀국수","이태원 쌀국수","이태원",
     weather={"available":True,"condition":"clear","temperature_c":31,"wind_speed_mps":2.0},
     checks=[("후보 존재", lambda p,t,r: len(r["candidates"])>0)])

print(f"{len(CASES)}개 케이스 정의 완료")

## 3. 전체 실행 + 결과 요약표

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 45)

def run_case(cs):
    parsed, task, res = run_search(cs["question"], cs["sq"], cs["loc"], cs["themes"], cs["filters"])
    if cs["weather"]:
        cands = rerank_with_weather(res["candidates"], cs["weather"], cs["question"], source_mode="rag_mcp")
        res = {**res, "candidates": cands}
    results = []
    for desc, fn in cs["checks"]:
        try:
            ok = bool(fn(parsed, task, res))
        except Exception as e:
            ok, desc = False, f"{desc} [예외:{type(e).__name__}]"
        results.append((desc, ok))
    return res, results

RUN = {}
rows, total, passed = [], 0, 0
fail_detail = []
for cs in CASES:
    res, results = run_case(cs)
    RUN[cs["id"]] = (cs, res, results)
    p = sum(1 for _, ok in results if ok)
    total += len(results); passed += p
    rows.append({"id": cs["id"], "유형": cs["type"], "질문": cs["question"][:26],
                 "추출": res["extracted_category"], "후보수": len(res["candidates"]),
                 "상위3": ", ".join(names(res)[:3]), "체크": f"{p}/{len(results)}",
                 "판정": "PASS" if p==len(results) else "FAIL"})
    for desc, ok in results:
        if not ok:
            fail_detail.append(f"#{cs['id']} {cs['type']}: {desc}")

summary = pd.DataFrame(rows)
display(summary)
print(f"\n총 체크 {total}건 / 통과 {passed}건 / 실패 {total-passed}건")
if fail_detail:
    print("\n실패 상세:")
    for f in fail_detail:
        print("  -", f)

## 4. 케이스별 점수 분해 상세

보고 싶은 케이스 번호를 `SHOW` 에 넣으면 벡터 3종 RRF / 부스팅 / 최종점수를 표로 본다.

In [ ]:
SHOW = [1, 3, 5, 19]   # 보고 싶은 케이스 id (자유 수정)

for cid in SHOW:
    cs, res, results = RUN[cid]
    print("=" * 90)
    print(f"#{cid} [{cs['type']}] {cs['question']}")
    print(f"  search_query={cs['sq']!r}  loc={cs['loc']!r}  filters={cs['filters']}  추출={res['extracted_category']}")
    for desc, ok in results:
        print(f"    [{'PASS' if ok else 'FAIL'}] {desc}")
    if res["candidates"]:
        display(score_table(res))
    else:
        print("  후보 없음")

## 5. 추가 진단 A - 카테고리 부스팅이 순위를 바꾸나

"부스트 적용" 순위 vs "부스트 제거(base만)" 순위를 비교한다.
둘이 같으면 부스팅이 순위에 영향을 주지 못하는 것이다.

In [ ]:
_, _, res = run_search("홍대 중식당", "홍대 중식당", "홍대")
cands = res["candidates"]
with_boost = [c["name"] for c in cands]
without = [c["name"] for c in sorted(cands, key=lambda c: (-base_of(c), -(c["review_count"] or 0), c["restaurant_id"]))]
from collections import Counter
print("status 분포:", dict(Counter(c["breakdown"]["category_status"] for c in cands)))
display(pd.DataFrame({"부스트 적용(실제)": with_boost, "부스트 제거 가정": without}))
print("순위 바뀜?", with_boost != without)
if with_boost == without:
    print("-> 부스팅이 순위에 영향 없음 (전원 match라 균일배율이 상쇄됨)")

## 6. 추가 진단 B - 날씨 재랭킹이 순위를 뒤집나

In [ ]:
_, _, res = run_search("내일 비 오는데 홍대 중식당", "홍대 중식당", "홍대")
before = [c["name"] for c in res["candidates"]]
rain = {"available": True, "condition": "rain", "temperature_c": 18, "wind_speed_mps": 9.0}
after = rerank_with_weather(copy.deepcopy(res["candidates"]), rain, "내일 비 오는데 홍대 중식당", source_mode="rag_mcp")
print("재랭킹 전:", before)
print("재랭킹 후:", [c["name"] for c in after])
print("순위 바뀜?", before != [c["name"] for c in after])
display(pd.DataFrame([{
    "이름": c["name"], "rag_score": round(c["rag_score"], 4),
    "weather_score": round(c["weather_score"], 3), "최종": round(c["score"], 4),
    "거리km": round(c["distance_km"], 2) if c["distance_km"] is not None else None,
    "주차": c["has_parking"],
    "날씨근거": " / ".join(c.get("weather_reasons") or []) or "-",
} for c in after]))

# RAG_ONLY 로 호출하면 날씨가 순서를 못 바꿔야 한다
guard = rerank_with_weather(copy.deepcopy(res["candidates"]), rain, "홍대 중식당", source_mode="rag_only")
print("\nRAG_ONLY 안전장치 - 순서 불변?", [c["restaurant_id"] for c in guard] == [c["restaurant_id"] for c in res["candidates"]])
print("RAG_ONLY - weather_score 전부 None?", all(c["weather_score"] is None for c in guard))

## 7. 추가 진단 C - '서울 홍대' 지역 오인

지역을 더 구체적으로 말할수록 결과가 사라지는지 확인한다.

In [ ]:
for label, loc in [("홍대만", "홍대"), ("서울 홍대", "서울 홍대")]:
    _, _, res = run_search(f"{loc} 중식당", f"{loc} 중식당", loc)
    print(f"[{label}] location={res['location_name']!r} origin=({res['origin_lat']}, {res['origin_lng']}) "
          f"후보 {len(res['candidates'])}건: {names(res)}")
print("\n'서울 홍대'가 '서울'로 잡혀 시청 반경 2km 밖이 되면 후보가 사라진다.")

## 참고

- 이 결과는 **가상 DB 22곳 + 실제 로직**으로 나온 것이다. 실제 DB 에서는 후보 수/점수가 달라진다.
- 파서(GPT) 품질(예: '서울 홍대'에서 GPT 가 location 을 뭘로 채우는지)은 이 노트북으로 알 수 없다.
  실제 OpenAI 키가 있는 환경에서 별도 진단 노트북으로 확인하라.
- 실패가 나는 항목(#11 등)은 현재 `query_policy.py`/`rag.py` 원본의 알려진 이슈다.
  패치를 적용하면 해당 케이스가 통과로 바뀐다.